# How big a GHZ state can Quobly's Pioneer hold together?

> **Billing notice:** the Quobly Alloy Forge emulator is currently **free to run** (0 credits).
>
> SENTINEL: Requires a qBraid API key; submits ~16 jobs and takes ~17 minutes to run live.

[Quobly's](https://quobly.io) **Pioneer** is a 10-qubit QPU built from electron spins in silicon, manufactured on standard CMOS processes. **Alloy Forge** is the physics-based emulator of it, reproducing the hardware's real error behaviour on a **linear nearest-neighbour** array of up to 15 qubits.

A GHZ state is a good stress test for that array. `(|0…0⟩ + |1…1⟩)/√2` is built by entangling each qubit with the next one along the chain, so growing it by one qubit adds exactly one more two-qubit interaction — and one more opportunity for the hardware to decohere. Sweeping the chain length turns the noise model into a curve.

This notebook runs that sweep and plots it. If you would rather not wait on ~16 sequential jobs, the results of a previous run are cached in `data/quobly_ghz_bench.json` and the plotting cells read from there.

In [ ]:
%%capture
%pip install 'qbraid[visualization]' qiskit matplotlib

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
device = provider.get_device("qbraid:quobly:sim:alloy-forge")

meta = device.metadata()
print(meta["name"], "|", meta["num_qubits"], "qubits |", meta["status"], "| pricing", meta["pricing"])

## Building GHZ along the coupling map

Pioneer's two-qubit interaction (`RZZ`) only exists between **adjacent** qubits. Writing the GHZ
ladder as a chain of `CX(i, i+1)` means every gate already names a neighbouring pair, so nothing
needs routing.

This matters: a two-qubit gate spanning non-adjacent qubits is accepted by the device and runs,
but the results come back on relabeled qubits and no longer line up with the indices you wrote.
Building along the chain sidesteps the problem entirely.

In [ ]:
from qiskit import QuantumCircuit


def ghz(n: int) -> QuantumCircuit:
    """GHZ state on n qubits, as a nearest-neighbour ladder."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)     # every pair is adjacent
    qc.measure_all()
    return qc


ghz(4).draw()

## Metrics

Results arrive carrying the full 10-bit Pioneer register, little-endian (qubit 0 rightmost), so the
first step is always to marginalise onto the qubits the circuit actually used.

Then two numbers per run:

- **GHZ population** — the share of shots landing in `|0…0⟩` or `|1…1⟩`. An ideal GHZ state gives 100%; everything else is error.
- **Hellinger fidelity** — how close the whole measured distribution is to the ideal 50/50, which also penalises an imbalance between the two peaks.

In [ ]:
import math


def marginalize(counts: dict, num_qubits: int) -> dict:
    """Trim padded Alloy Forge keys down to the circuit's own qubits."""
    out: dict = {}
    for bitstring, count in counts.items():
        key = bitstring[-num_qubits:]
        out[key] = out.get(key, 0) + count
    return out


def metrics(counts: dict, n: int, shots: int) -> dict:
    counts = marginalize(counts, n)
    zeros, ones = "0" * n, "1" * n
    ideal = {zeros: 0.5, ones: 0.5}
    ghz_pop = (counts.get(zeros, 0) + counts.get(ones, 0)) / shots
    hellinger = sum(
        math.sqrt(ideal.get(k, 0) * (v / shots)) for k, v in counts.items()
    ) ** 2
    return {"counts": counts, "ghz_pop": ghz_pop,
            "leakage": 1 - ghz_pop, "hellinger_fidelity": hellinger}

## The sweep

Each chain length runs twice: once with the noise model off, to confirm the circuit is right, and
once with it on.

**Two practical notes.** `noise` defaults to `True` on this device, so the noiseless run needs an
explicit opt-out — and every device option goes in a `runtime_options` dict rather than the `run()`
signature. A fixed `seed` makes noisy runs bit-for-bit reproducible.

Wall clock tracks `shots` more closely than circuit width (the same 4-qubit circuit takes ~41 s at
100 shots and ~168 s at 1000), and this device does not support batch submission, so the sweep is
sequential. At 200 shots the whole thing takes about 17 minutes.

Set `RUN_LIVE = True` to submit; leave it `False` to use the cached results.

In [ ]:
import json
import time
from pathlib import Path

RUN_LIVE = False
SHOTS = 200
SEED = 1234
CACHE = Path("data/quobly_ghz_bench.json")

if RUN_LIVE:
    rows = []
    for n in range(2, 10):
        for noise in (False, True):
            start = time.time()
            job = device.run(
                ghz(n), shots=SHOTS,
                runtime_options={"noise": noise, "seed": SEED},
            )
            counts = job.result().data.get_counts()
            rows.append({"n": n, "noise": noise, "shots": SHOTS,
                         "wall_s": round(time.time() - start, 1),
                         **metrics(counts, n, SHOTS)})
            print(f"GHZ-{n} noise={noise!s:5s}  "
                  f"{rows[-1]['ghz_pop']:6.1%}  {rows[-1]['wall_s']:5.1f}s")
else:
    rows = json.loads(CACHE.read_text())
    print(f"loaded {len(rows)} cached runs from {CACHE}")

In [ ]:
ns = sorted({r["n"] for r in rows})
clean = {r["n"]: r for r in rows if not r["noise"]}
noisy = {r["n"]: r for r in rows if r["noise"]}

print(f"{'chain':>6} {'noiseless':>11} {'Pioneer noise':>15} {'error pop.':>12}")
for n in ns:
    print(f"GHZ-{n:<2} {clean[n]['ghz_pop']:10.1%} {noisy[n]['ghz_pop']:15.1%} "
          f"{noisy[n]['leakage']:12.1%}")

## The curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(ns, [clean[n]["ghz_pop"] for n in ns], "-o", color="#2a78d6",
        linewidth=2, markersize=6, label="noise=False")
ax.plot(ns, [noisy[n]["ghz_pop"] for n in ns], "-o", color="#eb6834",
        linewidth=2, markersize=6, label="Pioneer noise model")

ax.set_xlabel("qubits in GHZ chain")
ax.set_ylabel("GHZ population")
ax.set_title("Noise accumulation on Quobly Alloy Forge "
             f"({rows[0]['shots']} shots, seed {SEED})")
ax.set_ylim(0, 1.05)
ax.set_xticks(ns)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The noiseless line is flat at 100% — the circuit is correct at every length, and the emulator
reproduces it exactly. The noisy line is the interesting one: a GHZ pair survives essentially
intact, and by nine qubits only about **30%** of shots still land on a valid GHZ outcome.

Each added qubit costs one more two-qubit interaction plus the idle time its neighbours spend
waiting, so the decay compounds. That shape — not any single number — is what a hardware noise
model is for.

In [ ]:
worst = min(noisy.values(), key=lambda r: r["ghz_pop"])
print(f"deepest chain measured : GHZ-{max(ns)}")
print(f"lowest GHZ population  : {worst['ghz_pop']:.1%} at GHZ-{worst['n']}")
print(f"total emulator time    : {sum(r['wall_s'] for r in rows) / 60:.1f} min "
      f"over {len(rows)} jobs")

## Seeing it fill in live, on the qBraid Agent Canvas

If you are running this inside [qBraid Lab](https://lab.qbraid.com), you can render the same figure
into the **Agent Canvas** — a panel beside your notebook or terminal — and rebuild it after each
job, so the curve draws itself while the sweep runs.

Write a self-contained HTML file and hand it to `qbraid-canvas`; the panel picks up changes within
about two seconds, so calling this after every result gives a live dashboard:

```python
import subprocess

def render(rows, path="quobly_canvas.html"):
    ...                                    # build a self-contained HTML document
    subprocess.run(["qbraid-canvas", path, "--title", "Quobly Alloy Forge benchmark"])
```

See `qbraid-canvas guide` in a Lab terminal for the full API, including the read-only
`window.qbraid.call('jobs.list')` bridge for embedding live job status in the page.

## Where next

- [Quobly Alloy Forge documentation](https://docs.qbraid.com/v2/sdk/user-guide/providers/native/quobly)
- `quobly_alloy_forge_quickstart.ipynb` — the five-minute quickstart for this device
- [Quobly](https://quobly.io) — silicon spin qubits on standard CMOS processes

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>